In [ ]:
from chemrar_retro import functions
from pydantic import BaseModel

In [ ]:
# snapshot
from typing import Any, Iterator
import pickle
from pathlib import Path
from typing import Hashable


class Snapshot:
    def __init__(self, obj, *, depth: int, file: Path | None = None) -> None:
        self.depth = depth
        self.dump_file = file

        if file:
            with open(file.as_posix(), "wb") as f:
                pickle.dump(
                    Snapshot._to_image(obj, depth),
                    file=f,
                    # protocol=pickle.HIGHEST_PROTOCOL,
                )
            self.dump = None
        else:
            self.dump = pickle.dumps(
                Snapshot._to_image(obj, depth),
                # protocol=pickle.HIGHEST_PROTOCOL,
            )

    def get_image(self):
        return self._get_image()

    def diff(self, obj, max_depth: int | None = None) -> dict[str, tuple]:
        depth = min(self.depth, max_depth or self.depth)
        a = self._get_image()

        diff = Snapshot._snapshot_diff(a, Snapshot._to_image(obj, max_depth=self.depth))
        result = {}
        for i, x0, x1 in diff:
            title = []

            for t in i.replace("]", "").split("[")[: depth + 1]:
                if t == "":
                    continue
                if t.startswith("'."):
                    title.append(t.replace("'", ""))
                else:
                    title.append(f"[{t}]")

            key = "".join(title).strip()

            # key = "".join(
            #                 f"['{t}']" if t and not t.startswith(".") else t

            #             ).strip()

            result.update({key: (x0, x1)})

        print("\n".join(f"{e:3}|{i}" for e, i in enumerate(result)))
        return result

    def _get_image(self) -> dict:  # type: ignore
        if self.dump_file:
            with open(self.dump_file, "rb") as f:
                return pickle.load(f)
        elif self.dump:
            return pickle.loads(self.dump)

    @staticmethod
    def _to_image(
        obj: object,
        max_depth: int,
        depth: int = 0,
        memo: frozenset[int] = frozenset(),
    ) -> object:

        # print(type(obj),obj)
        if depth > max_depth:
            return _DepthLimited(type(obj).__name__)

        if id(obj) in memo:
            return _Skipped("cycle")

        if isinstance(obj, dict):
            memo = memo | {id(obj)}
            if len(obj) > 500:
                # print("long dict")
                return _Skipped("long dict")
            # print(obj, depth, max_depth)

            return {
                k if isinstance(k, Hashable) else "@nonHashable@": Snapshot._to_image(
                    v, max_depth, depth + 1, memo
                )
                if isinstance(k, str)
                else (
                    Snapshot._to_image(k, max_depth, depth + 1, memo),
                    Snapshot._to_image(v, max_depth, depth + 1, memo),
                )
                for k, v in obj.items()
            }

        if isinstance(obj, (list, tuple, set)):
            memo = memo | {id(obj)}
            if len(obj) > 500:
                # print("long sequence")

                return _Skipped("long sequence")
            return [Snapshot._to_image(v, max_depth, depth + 1, memo) for v in obj]

        if isinstance(obj, (pd.DataFrame, pd.Series, np.ndarray)):
            return _Skipped("data")

        # if isinstance(obj, BaseModel):
        #     memo = memo | {id(obj)}
        #     return {
        #         f".{k}": Snapshot._sanitize(v, max_depth, depth + 1, memo)
        #         for k, v in obj.model_dump().items()
        #     }

        # print(type(obj),obj)
        # try:
        #     pickle.dumps(obj)
        #     return obj
        # except Exception:
        #     pass

        # print(type(obj), obj)

        attrs: dict[str, object] = {}
        if hasattr(obj, "__dict__"):
            attrs.update(vars(obj))
        for name in getattr(type(obj), "__slots__", ()):
            if hasattr(obj, name):
                attrs[name] = getattr(obj, name)

        if attrs:
            return {
                "__type__": type(obj).__name__,
                **{
                    f".{k}": Snapshot._to_image(v, max_depth, depth + 1, memo)
                    for k, v in attrs.items()
                },
            }

        return repr(obj)

    @staticmethod
    def _snapshot_diff(a: Any, b: Any, path: str = "") -> Iterator[tuple[str, Any, Any]]:
        if type(a) is not type(b):
            yield path, a, b
            return

        if isinstance(a, dict):
            for key in a.keys() | b.keys():
                if key not in a:
                    yield f"{path}[{key!r}]", "<missing>", b[key]
                elif key not in b:
                    yield f"{path}[{key!r}]", a[key], "<missing>"
                else:
                    yield from Snapshot._snapshot_diff(a[key], b[key], f"{path}[{key!r}]")
            return

        if isinstance(a, (list, tuple)):
            if len(a) != len(b):
                yield f"{path}.len", len(a), len(b)
                return
            for i, (av, bv) in enumerate(zip(a, b)):
                yield from Snapshot._snapshot_diff(av, bv, f"{path}[{i}]")
            return

        if isinstance(a, set):
            if a != b:
                yield path, a - b, b - a
            return
        try:
            if a != b:
                yield path, a, b
        except:
            if all(a != b):
                yield path, a, b


class _Skipped:
    __slots__ = ("type_name",)

    def __init__(self, type_name: str) -> None:
        self.type_name = type_name

    def __repr__(self) -> str:
        return f"<skipped:{self.type_name}>"

    def __hash__(self) -> int:
        return hash(self.type_name + "_Skipped")

    def __eq__(self, other: object) -> bool:
        return isinstance(other, _Skipped) and self.type_name == other.type_name


class _DepthLimited:
    __slots__ = ("type_name",)

    def __init__(self, type_name: str) -> None:
        self.type_name = type_name

    def __repr__(self) -> str:
        return f"<depth-limit:{self.type_name}>"

    def __hash__(self) -> int:
        return hash(self.type_name + "_DepthLimited")

    def __eq__(self, other: object) -> bool:
        return isinstance(other, _DepthLimited) and self.type_name == other.type_name


In [ ]:
class A(BaseModel):
    a: int = 1
    b: dict = dict(x=0, y=8)
    c: list = [1, 2]


a1 = A()
a2 = A(
    a=1,
    b=dict(x=0, y=7),
    c=[1, 3],
)

s1 = Snapshot(a1, depth=10)
diff = s1.diff(a2)

In [ ]:
def imutate_func(
    engine: functions.Engine,
    smiles: str,
):
    engine = functions._copy_engine(engine)

    engine.target_smiles = smiles
    engine.tree_search()
    engine.build_routes()
    return


def mutate_func(
    engine: functions.Engine,
    smiles: str,
):
    engine.target_smiles = smiles

    engine.tree_search(show_progress=False)
    engine.build_routes()
    return


In [ ]:
[i for i in diff.keys() if "logger" not in i]